# BƯỚC 1: THU THẬP DỮ LIỆU (DATA INGESTION)

**Mục tiêu:** Thu thập dữ liệu lịch sử OHLCV (Open, High, Low, Close, Volume) cho ít nhất 3 mã cổ phiếu niêm yết và dữ liệu chỉ số thị trường bổ sung.

**Yêu cầu kỹ thuật:**
- Phạm vi thời gian: Từ 01/01/2020 đến hiện tại.
- Lưu dữ liệu thô ra file `.csv` trước khi tiến hành xử lý.

## 1.1 Khởi tạo Môi trường và Thư viện

Import các thư viện cần thiết như `pandas`, `requests`, `matplotlib` và tự động tạo cấu trúc thư mục `data/raw/` để lưu trữ dữ liệu thô.

In [2]:
import requests
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import os
from vnstock import stock_historical_data

#Cấu hình giao diện biểu đồ
sns.set_theme(style="whitegrid")

#Tạo cấu trúc thư mục để lưu dữ liệu thô
os.makedirs('../data/raw', exist_ok=True)

## 1.2 Khai báo Tham số Hệ thống

Thiết lập danh sách các mã cổ phiếu cần phân tích (VNM, FPT, VIC) và quy định phạm vi thời gian thu thập dữ liệu (từ 01/01/2020 đến nay). Định dạng ngày tháng được chuyển đổi sang chuẩn timestamp để tương thích với API.

In [ ]:
#Danh sách 3 mã cổ phiếu niêm yết
tickers = ['VNM', 'FPT', 'VIC']

#Khai báo thời gian theo định dạng chuỗi
start_date = '2020-01-01'
end_date = datetime.today().strftime('%Y-%m-%d')

#Chuyển đổi ngày sang định dạng timestamp (giây) để gọi API Entrade
start_ts = int(datetime.strptime(start_date, '%Y-%m-%d').timestamp())
end_ts = int(datetime.strptime(end_date, '%Y-%m-%d').timestamp())

#Dictionary lưu trữ các DataFrame để dùng cho các cell sau
data_frames = {} 


## 1.3 Thu thập Dữ liệu Lịch sử Cổ phiếu

Sử dụng phương pháp gọi API trực tiếp để đảm bảo tính ổn định cao nhất khi lấy dữ liệu OHLCV (mở cửa, cao nhất, thấp nhất, đóng cửa, khối lượng giao dịch). Dữ liệu sau khi tải về sẽ được định dạng lại Index và lưu thành các file `_raw.csv`.

In [ ]:
for ticker in tickers:
    try:
        #Lấy data
        df = stock_historical_data(ticker, start_date, end_date, '1D', 'stock')

        #Check nhanh và bỏ qua nếu không có dữ liệu
        if df is None or df.empty:
            print(f"[{ticker}] Không có dữ liệu, skip.")
            continue
        
        #Format cột
        df = df.rename(columns={
            'time': 'Date', 'open': 'Open', 'high': 'High', 
            'low': 'Low', 'close': 'Close', 'volume': 'Volume'
        })
        
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
        df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
        
        #Lưu file
        file_path = f"../data/raw/{ticker}_raw.csv"
        df.to_csv(file_path)
        data_frames[ticker] = df
        
        print(f"-> Xong {ticker} ({len(df)} rows)")
        
    except Exception as e:
        print(f"Lỗi ở mã {ticker}: {e}")

## 1.4 Thu thập Dữ liệu Bổ sung

Tiến hành tải thêm dữ liệu của chỉ số thị trường chung VN-Index theo ngày. Dữ liệu này sẽ làm cơ sở để đối chiếu xu hướng của các mã cổ phiếu riêng lẻ với toàn bộ thị trường.

In [ ]:
# --- Lấy dữ liệu VNINDEX ---
try:
    df_vn = stock_historical_data('VNINDEX', start_date, end_date, '1D', 'index')
    
    if df_vn is not None and not df_vn.empty:
        #Chuẩn hóa format
        df_vn = df_vn.rename(columns={
            'time': 'Date', 'open': 'Open', 'high': 'High', 
            'low': 'Low', 'close': 'Close', 'volume': 'Volume'
        })
        df_vn['Date'] = pd.to_datetime(df_vn['Date'])
        df_vn.set_index('Date', inplace=True)
        df_vn = df_vn[['Open', 'High', 'Low', 'Close', 'Volume']]
        
        #Lưu trữ
        df_vn.to_csv("../data/raw/VNINDEX_raw.csv")
        data_frames['VNINDEX'] = df_vn
        print(f"-> Xong VNINDEX ({len(df_vn)} rows)")
    else:
        print("[VNINDEX] Rỗng hoặc không có dữ liệu.")
        
except Exception as e:
    print(f"Lỗi tải VNINDEX: {e}")


# --- Lấy tỷ giá USD/VND ---
try:
    df_usd = yf.download('USDVND=X', start=start_date, end=end_date, progress=False)
    
    if not df_usd.empty:
        #Xử lý lỗi MultiIndex của bản yfinance mới
        if isinstance(df_usd.columns, pd.MultiIndex):
            df_usd.columns = df_usd.columns.droplevel(1)
            
        df_usd.index.name = 'Date'
        
        #Bỏ cột thừa
        if 'Adj Close' in df_usd.columns:
            df_usd.drop(columns=['Adj Close'], inplace=True)
            
        #Lưu trữ
        df_usd.to_csv("../data/raw/USD_VND_raw.csv")
        data_frames['USD_VND'] = df_usd
        print(f"-> Xong USD/VND ({len(df_usd)} rows)")
    else:
        print("[USD_VND] Không có dữ liệu.")
        
except Exception as e:
    print(f"Lỗi tải USD_VND: {e}")

## 1.5 Khám phá và Kiểm tra Dữ liệu (EDA Cơ bản)

Hiển thị thông tin cấu trúc DataFrame (`df.info()`) và các thống kê mô tả cơ bản (`df.describe()`) của các mã cổ phiếu nhằm kiểm tra nhanh tính toàn vẹn của dữ liệu trước khi chuyển sang bước tiền xử lý.

In [ ]:
for ticker in tickers:
    #Bỏ qua nhanh nếu không có dữ liệu
    if ticker not in data_frames:
        print(f"[{ticker}] Không có data, skip.")
        continue
    df = data_frames[ticker]
    print(f"\n========== {ticker} ==========")
    df.info()
    display(df.describe())

## 1.6 Trực quan hóa Dữ liệu

Vẽ biểu đồ xu hướng biến động của giá đóng cửa (Close Price) của các mã cổ phiếu trong giai đoạn từ năm 2020 đến nay. Biểu đồ giúp nhận diện trực quan các chu kỳ tăng/giảm và những vùng biến động mạnh.

In [ ]:
#Kiểm tra nhanh xem có data không
if not data_frames:
    print("Không có dữ liệu để vẽ.")
else:
    plt.figure(figsize=(14, 7))
    
    for ticker, df in data_frames.items():
        plt.plot(df.index, df['Close'], label=ticker)

    #Tạo tiêu đề
    tickers_str = ", ".join(data_frames.keys())
    plt.title(f'Biểu đồ Giá Đóng Cửa - {tickers_str}', fontsize=14, fontweight='bold')
    
    plt.xlabel('Thời gian')
    plt.ylabel('Giá đóng cửa (VND)')
    plt.legend()
    plt.tight_layout()
    
    #Lưu biểu đồ
    os.makedirs('../outputs', exist_ok=True)
    plt.savefig('../outputs/01_closing_price_chart.png', dpi=300)
    plt.show()